# Exploratory Data Analytics

In [1]:
import pandas as pd, numpy as np, plotly,os

In [2]:
cur_dir = os.path.abspath('')
dataset_path = os.path.join(cur_dir,'..','Datasets/online+retail/Processed/online_retail_joined.pkl')

df = pd.read_pickle(dataset_path)

In [3]:
df

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
...,...,...,...,...,...,...,...,...
1067365,538171,22271,FELTCRAFT DOLL ROSIE,2,2010-12-09 20:01:00,2.95,17530.0,United Kingdom
1067366,538171,22750,FELTCRAFT PRINCESS LOLA DOLL,1,2010-12-09 20:01:00,3.75,17530.0,United Kingdom
1067367,538171,22751,FELTCRAFT PRINCESS OLIVIA DOLL,1,2010-12-09 20:01:00,3.75,17530.0,United Kingdom
1067368,538171,20970,PINK FLORAL FELTCRAFT SHOULDER BAG,2,2010-12-09 20:01:00,3.75,17530.0,United Kingdom


### RFM Analysis
Using those who have multiple purchases (Loyal customers)
>1. Recency
2. Frequency
3. Monetary value

In [4]:
loyal_customers_df = df[df.CustomerID!=-1]

In [5]:
customer_recency = loyal_customers_df.groupby("CustomerID",as_index=False)["InvoiceDate"].max()
customer_recency.columns = ["CustomerID","LastPurchaseDate"]
customer_recency["LastPurchaseDate"] = customer_recency["LastPurchaseDate"].dt.normalize()

latest_purchase_date = customer_recency["LastPurchaseDate"].max()

customer_recency["Recency"] = customer_recency["LastPurchaseDate"].apply(
    lambda x: (latest_purchase_date - x).days)

In [6]:
customer_recency.head()

,CustomerID,LastPurchaseDate,Recency
0,12346.0,2011-01-18,325
1,12347.0,2011-12-07,2
2,12348.0,2011-09-25,75
3,12349.0,2011-11-21,18
4,12350.0,2011-02-02,310


In [7]:
customer_freq = loyal_customers_df.groupby("CustomerID",as_index=False)["InvoiceDate"].nunique()
customer_freq.columns = ["CustomerID", "Frequency"]

In [8]:
customer_freq.head()

,CustomerID,Frequency
0,12346.0,17
1,12347.0,8
2,12348.0,5
3,12349.0,5
4,12350.0,1


#### Monetary
- As we see, there are instances where total spending is Negative. This is suspicious.

In [9]:
loyal_customers_df["TotalPrice"] = loyal_customers_df["UnitPrice"]*loyal_customers_df["Quantity"]

customer_monetary = loyal_customers_df.groupby("CustomerID",as_index=False)["TotalPrice"].sum()
customer_monetary.columns = ["CustomerID", "Monetary"]

In [10]:
customer_monetary.head()

,CustomerID,Monetary
0,12346.0,-64.68
1,12347.0,5598.52
2,12348.0,2019.40
3,12349.0,4270.48
4,12350.0,314.60


##### We check that every returned item corresponds correctly to the previous purchase.

In [11]:
returned = loyal_customers_df.loc[loyal_customers_df["Quantity"]<0,("StockCode", "Quantity", "UnitPrice", "CustomerID","InvoiceDate")]
purchases = loyal_customers_df.loc[loyal_customers_df["Quantity"]>0,("StockCode", "Quantity", "UnitPrice", "CustomerID","InvoiceDate")]

returned.InvoiceDate = returned.InvoiceDate.dt.normalize()
purchases.InvoiceDate = purchases.InvoiceDate.dt.normalize()

In [12]:
returned["Quantity"] = -returned["Quantity"]
purchases["Quantity"] = purchases["Quantity"]

A valid return is one where the returned quantity is less than purchased, and the return date is after the purchase.

In [13]:
returned = (returned.groupby(["StockCode","CustomerID","InvoiceDate"])["Quantity"].sum()).reset_index()
purchases = (purchases.groupby(["StockCode","CustomerID","InvoiceDate"])["Quantity"].sum()).reset_index()

In [14]:
matches = returned.merge(
    purchases,
    how='inner',
    on=["StockCode","CustomerID"],
    suffixes=("_return", "_purchase")
)

In [15]:
valid_matches = matches[
    (matches["InvoiceDate_purchase"] <= matches["InvoiceDate_return"])
    ]

In [16]:
valid_return = (valid_matches.groupby(["StockCode","CustomerID","Quantity_return"])["Quantity_purchase"].sum()).reset_index()

In [17]:
invalid_return = valid_return[valid_return["Quantity_purchase"]<valid_return["Quantity_return"]]

Based on these invalid returns, we remove all the records of stock,id,quantity,... based on the returned date.

In [18]:
remove_index = (valid_matches.StockCode.isin(invalid_return.StockCode))&\
                (valid_matches.CustomerID.isin(invalid_return.CustomerID))

invalid_matches = valid_matches[remove_index]

In [19]:
invalid_matches = invalid_matches[["StockCode","CustomerID","InvoiceDate_return"]]

In [20]:
invalid_matches

,StockCode,CustomerID,InvoiceDate_return
438,15058C,15581.0,2010-07-01
468,16054,16900.0,2011-11-28
505,16202E,15916.0,2010-12-08
705,20675,13069.0,2011-04-14
706,20675,13069.0,2011-04-14
...,...,...,...
63840,M,17949.0,2010-01-29
63842,M,17949.0,2010-04-15
63844,M,17949.0,2011-05-23
63846,M,17949.0,2011-10-24


In [21]:
loyal_customers_df = loyal_customers_df.reset_index(names="orig_idx")

dummy_df = invalid_matches.merge(
    loyal_customers_df,
    on=["StockCode","CustomerID"],
    how="inner"
)

to_remove = dummy_df.loc[dummy_df["InvoiceDate"]<=dummy_df["InvoiceDate_return"],"orig_idx"].unique()

In [22]:
loyal_customers_df = loyal_customers_df.loc[~(loyal_customers_df.orig_idx.isin(to_remove))]

In [23]:
returned = loyal_customers_df.loc[loyal_customers_df["Quantity"]<0,("StockCode", "Quantity", "UnitPrice", "CustomerID","InvoiceDate")]
purchases = loyal_customers_df.loc[loyal_customers_df["Quantity"]>0,("StockCode", "Quantity", "UnitPrice", "CustomerID","InvoiceDate")]

returned.InvoiceDate = returned.InvoiceDate.dt.normalize()
purchases.InvoiceDate = purchases.InvoiceDate.dt.normalize()

In [24]:
returned["Quantity"] = -returned["Quantity"]
purchases["Quantity"] = purchases["Quantity"]

A valid return is one where the returned quantity is less than purchased, and the return date is after the purchase.

In [25]:
returned = (returned.groupby(["StockCode","CustomerID","InvoiceDate"])["Quantity"].sum()).reset_index()
purchases = (purchases.groupby(["StockCode","CustomerID","InvoiceDate"])["Quantity"].sum()).reset_index()

In [26]:
matches = returned.merge(
    purchases,
    how='inner',
    on=["StockCode","CustomerID"],
    suffixes=("_return", "_purchase")
)

In [27]:
valid_matches = matches[
    (matches["InvoiceDate_purchase"] <= matches["InvoiceDate_return"])
    ]

In [28]:
valid_return = (valid_matches.groupby(["StockCode","CustomerID","Quantity_return"])["Quantity_purchase"].sum()).reset_index()

In [29]:
invalid_return = valid_return[valid_return["Quantity_purchase"]<valid_return["Quantity_return"]]

Based on these invalid returns, we remove all the records of stock,id,quantity,... based on the returned date.

In [30]:
remove_index = (valid_matches.StockCode.isin(invalid_return.StockCode))&\
                (valid_matches.CustomerID.isin(invalid_return.CustomerID))

invalid_matches = valid_matches[remove_index]

In [31]:
invalid_matches = invalid_matches[["StockCode","CustomerID","InvoiceDate_return"]]

In [32]:
invalid_matches

,StockCode,CustomerID,InvoiceDate_return
10264,21314,13440.0,2010-09-21
10737,21314,17949.0,2009-12-03
12264,21530,14590.0,2010-10-11
35517,22567,17894.0,2010-11-22
47572,35004G,14031.0,2009-12-18
49596,47504J,14191.0,2009-12-11
50951,75049L,14667.0,2010-01-14
52211,79323W,13799.0,2010-05-14
58384,85099B,12748.0,2011-08-03
58703,85099B,14667.0,2011-02-20
